In [1]:
import nltk
# nltk.download('averaged_perceptron_tagger')
import traceback
import os
import chardet
import magic
from langchain_docling import DoclingLoader
from langchain_unstructured.document_loaders import UnstructuredLoader
from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader,
    PyMuPDFLoader,
    UnstructuredPowerPointLoader,
    Docx2txtLoader,
    DataFrameLoader,
    CSVLoader,
    UnstructuredCSVLoader,
    PolarsDataFrameLoader,
    UnstructuredExcelLoader,
    UnstructuredTSVLoader,
    
)

#MIME 타입별 Loader 매핑
MIME_LOADER_ROUTER = {
    # 텍스트
    "text/plain": TextLoader,
    "text/markdown": TextLoader,
    "application/rtf": UnstructuredLoader,  # 또는 RTF 전용 Loader
    "text/rtf": UnstructuredLoader,
    "text/x-org": UnstructuredLoader,       # org-mode 전용 Loader가 있으면 교체
    "application/onenote": UnstructuredLoader,  # .one 전용 Loader가 있으면 교체

    # 문서
    "application/pdf": PyMuPDFLoader,
    "application/vnd.openxmlformats-officedocument.wordprocessingml.document": Docx2txtLoader,
    "application/msword": UnstructuredLoader,  # .doc 전용 Loader가 있으면 교체
    "application/vnd.ms-word.document.macroEnabled.12": UnstructuredLoader,
    "application/x-iwork-pages-sffpages": UnstructuredLoader,
    "application/vnd.apple.pages": UnstructuredLoader,
    "application/x-hwp": UnstructuredLoader,  # hwp 전용 Loader가 있으면 교체
    "application/haansofthwp": UnstructuredLoader,

    # 스프레드시트
    "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet": UnstructuredExcelLoader,
    "application/vnd.ms-excel": CSVLoader,
    "application/vnd.ms-office": CSVLoader,
    "text/csv": CSVLoader,
    "application/csv": CSVLoader,
    "text/tab-separated-values": UnstructuredTSVLoader,
    "application/x-iwork-numbers-sffnumbers": UnstructuredLoader,
    "application/vnd.apple.numbers": UnstructuredLoader,

    # 프레젠테이션
    "application/vnd.openxmlformats-officedocument.presentationml.presentation": DoclingLoader,
    "application/vnd.ms-powerpoint": DoclingLoader,
    "application/mspowerpoint": DoclingLoader,
    "application/x-iwork-keynote-sffkey": UnstructuredLoader,
    "application/vnd.apple.keynote": UnstructuredLoader,

    # 기타
    "application/octet-stream": UnstructuredLoader,  # 미확인/바이너리 파일 폴백
}

#UnstructuredLoader를 사용하는 MIME 타입
UNSTRUCTURED_MIME_TYPES = {
    "application/msword",
    "application/vnd.ms-word.document.macroEnabled.12",
    "application/x-iwork-pages-sffpages",
    "application/vnd.apple.pages",
    "application/x-hwp",
    "application/haansofthwp",
    "application/x-iwork-numbers-sffnumbers",
    "application/vnd.apple.numbers",
    "application/x-iwork-keynote-sffkey",
    "application/vnd.apple.keynote",
    "application/CDFV2",
    "application/octet-stream",
    "application/vnd.ms-powerpoint"
}

# 텍스트 MIME 타입 정의
TEXT_MIME_TYPES = {
    "text/plain",
    "text/markdown",
    "text/csv",
    "application/csv",
    "text/tab-separated-values",
    "application/json",
    "application/xml",
    "text/xml",
    "text/html",
    "application/javascript",
    "application/x-javascript",
    "text/x-python",
    "text/x-c",
    "text/x-c++",
    "text/x-java-source",
    "text/x-shellscript",
    "text/x-org",
    "text/rtf",
    "application/rtf",
}

# 파일 인코딩 반환 함수
def find_encoding(path, num_bytes=10000):
    with open(path, 'rb') as f:
        rawdata = f.read(num_bytes)
    result = chardet.detect(rawdata)
    return result['encoding'] if result['encoding'] else 'utf-8'

# 파일 MIME 타입 반환 함수
def get_mime_type(path):
    mime = magic.Magic(mime=True)
    return mime.from_file(path)

# 파일별 Loader 선택 (MIME 기반)
def get_loader(path, mime_loader_router=MIME_LOADER_ROUTER, fallback_loader=UnstructuredLoader):
    # 숨김/시스템 파일 패스
    if os.path.basename(path).startswith('.'):
        return None
    mime_type = get_mime_type(path)
    loader_cls = mime_loader_router.get(mime_type, fallback_loader)
    if mime_type in TEXT_MIME_TYPES:
        encoding = find_encoding(path)
        return loader_cls(path, encoding=encoding)
    return loader_cls(path)

# 폴더 내 모든 파일 경로 수집
def get_file_paths(folder_path):
    file_paths = []
    for file in os.listdir(folder_path):
        full_path = os.path.join(folder_path, file)
        if os.path.isfile(full_path):
            file_paths.append(full_path)
    return file_paths

def connect_loader(folder_path):
    docs = [] # 정상적으로 로드된 파일 리스트
    failed_files = [] # 로더 실행 중 에러가 난 파일 리스트
    deferred_files = [] # UnstructuredLoader로 분리된 파일 리스트
    unsupported_files = [] # 라우터에 없는 MIME 파일 리스트

    file_paths = get_file_paths(folder_path)
    for file_path in file_paths:
        mime_type = get_mime_type(file_path)
        if mime_type in UNSTRUCTURED_MIME_TYPES:
            print(f"[DEFERRED] {mime_type} -> {file_path}")
            deferred_files.append(file_path)
            continue

        loader_cls = MIME_LOADER_ROUTER.get(mime_type)
        if loader_cls is None:
            print(f"[UNSUPPORTED] {mime_type} -> {file_path}")
            unsupported_files.append(file_path)
            continue

        try:
            loader = get_loader(file_path)
            if loader is None:
                continue
            loaded = loader.load()
            docs.extend(loaded)
        except Exception as e:
            print(f"[ERROR] 파일 로드 실패: {file_path} ({type(e).__name__}) - {e}")
            traceback.print_exc()
            failed_files.append(file_path)
            continue
    return docs, failed_files, deferred_files, unsupported_files


/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
# documents/doc

docs, failed_files, deferred_files, unsupported_files = connect_loader('../documents/doc')

[
    f"정상적으로 로드된 문서(Document) 개수: {len(docs)}",
    f"로더 실행 중 에러가 발생한 파일 개수: {len(failed_files)}",
    f"UnstructuredLoader로 분리(추후 처리)된 파일 개수: {len(deferred_files)}",
    f"지원하지 않는 MIME 타입(라우터 미등록) 파일 개수: {len(unsupported_files)}",
]

[DEFERRED] application/msword -> ../documents/doc/Word6_sections.doc
[DEFERRED] application/msword -> ../documents/doc/Bug50075.doc
[DEFERRED] application/msword -> ../documents/doc/64132.doc
[DEFERRED] application/msword -> ../documents/doc/word95err.doc
[DEFERRED] application/msword -> ../documents/doc/47304.doc
[DEFERRED] application/msword -> ../documents/doc/SampleDoc.doc
[DEFERRED] application/msword -> ../documents/doc/57843.doc
[UNSUPPORTED] application/x-ole-storage -> ../documents/doc/clusterfuzz-testcase-minimized-POIHWPFFuzzer-5195207308541952.doc
[UNSUPPORTED] application/x-ole-storage -> ../documents/doc/clusterfuzz-testcase-minimized-POIHWPFFuzzer-5050208641482752.doc
[DEFERRED] application/msword -> ../documents/doc/cn.orthodox.www_divenbog_APRIL_30-APRIL.DOC
[DEFERRED] application/msword -> ../documents/doc/clusterfuzz-testcase-minimized-POIHWPFFuzzer-5074346559012864.doc
[ERROR] 파일 로드 실패: ../documents/doc/clusterfuzz-testcase-minimized-POIXWPFFuzzer-5166796835258368.d

Traceback (most recent call last):
  File "/var/folders/v8/f7vcy2kn4tq3lw_bpdwqbs9m0000gn/T/ipykernel_47817/4185757274.py", line 162, in connect_loader
    loaded = loader.load()
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/langchain_community/document_loaders/word_document.py", line 61, in load
    page_content=docx2txt.process(self.file_path),
                 ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/docx2txt/docx2txt.py", line 76, in process
    zipf = zipfile.ZipFile(docx)
  File "/Users/wonsik/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/zipfile/__init__.py", line 1385, in __init__
    self._RealGetContents()
    ~~~~~~~~~~~~~~~~~~~~~^^
  File "/Users/wonsik/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/zipfile/__init__.py", line 1482, in _RealGetContents
    raise BadZipFile("Bad magic number for 

[DEFERRED] application/msword -> ../documents/doc/page-break.doc
[DEFERRED] application/msword -> ../documents/doc/Bug34898.doc
[DEFERRED] application/msword -> ../documents/doc/picture.doc
[UNSUPPORTED] image/jpeg -> ../documents/doc/nature1.jpg
[DEFERRED] application/octet-stream -> ../documents/doc/table-indent.docx
[ERROR] 파일 로드 실패: ../documents/doc/clusterfuzz-testcase-minimized-POIXWPFFuzzer-6442791109263360.docx (BadZipFile) - Bad magic number for central directory
[DEFERRED] application/msword -> ../documents/doc/documentProperties.doc
[DEFERRED] application/msword -> ../documents/doc/page-break-before.doc
[UNSUPPORTED] application/x-ole-storage -> ../documents/doc/clusterfuzz-testcase-POIHWPFFuzzer-5696094627495936.doc
[DEFERRED] application/msword -> ../documents/doc/Bug44603.doc
[DEFERRED] application/msword -> ../documents/doc/HeaderFooterProblematic.doc
[DEFERRED] application/msword -> ../documents/doc/ThreeColHeadFoot.doc
[UNSUPPORTED] image/jpeg -> ../documents/doc/natur

Traceback (most recent call last):
  File "/var/folders/v8/f7vcy2kn4tq3lw_bpdwqbs9m0000gn/T/ipykernel_47817/4185757274.py", line 162, in connect_loader
    loaded = loader.load()
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/langchain_community/document_loaders/word_document.py", line 61, in load
    page_content=docx2txt.process(self.file_path),
                 ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/docx2txt/docx2txt.py", line 76, in process
    zipf = zipfile.ZipFile(docx)
  File "/Users/wonsik/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/zipfile/__init__.py", line 1385, in __init__
    self._RealGetContents()
    ~~~~~~~~~~~~~~~~~~~~~^^
  File "/Users/wonsik/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/zipfile/__init__.py", line 1452, in _RealGetContents
    raise BadZipFile("File is not a zip fil

[DEFERRED] application/msword -> ../documents/doc/equation.doc
[DEFERRED] application/msword -> ../documents/doc/Bug47742.doc
[DEFERRED] application/msword -> ../documents/doc/Lists.doc
[DEFERRED] application/msword -> ../documents/doc/Bug48065.doc
[DEFERRED] application/msword -> ../documents/doc/60279.doc
[DEFERRED] application/msword -> ../documents/doc/testPictures.doc
[DEFERRED] application/msword -> ../documents/doc/Bug50936_3.doc
[DEFERRED] application/msword -> ../documents/doc/Bug50936_2.doc
[ERROR] 파일 로드 실패: ../documents/doc/clusterfuzz-testcase-minimized-POIXWPFFuzzer-5313273089884160.docx (BadZipFile) - File is not a zip file
[DEFERRED] application/msword -> ../documents/doc/ProblemExtracting.doc
[DEFERRED] application/msword -> ../documents/doc/cpansearch.perl.org_src_tobyink_acme-rundoc-0.001_word-lib_hello_world.docm
[DEFERRED] application/octet-stream -> ../documents/doc/table-alignment.docx
[DEFERRED] application/msword -> ../documents/doc/Bug53380_1.doc
[DEFERRED] app

...

[DEFERRED] application/msword -> ../documents/doc/ThreeColHead.doc
[DEFERRED] application/msword -> ../documents/doc/SimpleHeadThreeColFoot.doc
[DEFERRED] application/msword -> ../documents/doc/bug65255.doc
[DEFERRED] application/msword -> ../documents/doc/Word6.doc
[DEFERRED] application/msword -> ../documents/doc/Bug53380_3.doc
[DEFERRED] application/msword -> ../documents/doc/ThreeColFoot.doc
[DEFERRED] application/msword -> ../documents/doc/Bug50936_1.doc
[DEFERRED] application/msword -> ../documents/doc/cap.stanford.edu_profiles_viewbiosketch_facultyid=4009&name=m_maciver.doc
[DEFERRED] application/msword -> ../documents/doc/Bug53380_2.doc
[ERROR] 파일 로드 실패: ../documents/doc/clusterfuzz-testcase-minimized-POIFuzzer-6709287337197568.docx (BadZipFile) - Bad magic number for central directory
[DEFERRED] application/msword -> ../documents/doc/two_images.doc
[DEFERRED] application/msword -> ../documents/doc/footnote.doc
[DEFERRED] application/msword -> ../documents/doc/empty.doc
[DEFERR

Traceback (most recent call last):
  File "/var/folders/v8/f7vcy2kn4tq3lw_bpdwqbs9m0000gn/T/ipykernel_47817/4185757274.py", line 162, in connect_loader
    loaded = loader.load()
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/langchain_community/document_loaders/word_document.py", line 61, in load
    page_content=docx2txt.process(self.file_path),
                 ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/docx2txt/docx2txt.py", line 76, in process
    zipf = zipfile.ZipFile(docx)
  File "/Users/wonsik/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/zipfile/__init__.py", line 1385, in __init__
    self._RealGetContents()
    ~~~~~~~~~~~~~~~~~~~~~^^
  File "/Users/wonsik/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/zipfile/__init__.py", line 1482, in _RealGetContents
    raise BadZipFile("Bad magic number for 

[DEFERRED] application/msword -> ../documents/doc/59322.doc
[DEFERRED] application/msword -> ../documents/doc/testCroppedPictures.doc
[UNSUPPORTED] image/jpeg -> ../documents/doc/abstract4.jpg
[DEFERRED] application/msword -> ../documents/doc/61911.doc
[DEFERRED] application/msword -> ../documents/doc/Bug49933.doc
[ERROR] 파일 로드 실패: ../documents/doc/ExternalEntityInText.docx (ParseError) - undefined entity &testent;: line 5, column 1180
[DEFERRED] application/msword -> ../documents/doc/ListEntryNoListTable.doc
[DEFERRED] application/msword -> ../documents/doc/innertable.doc
[DEFERRED] application/msword -> ../documents/doc/52117.doc
[DEFERRED] application/msword -> ../documents/doc/MSWriteOld.wri
[DEFERRED] application/msword -> ../documents/doc/Bug49919.doc
[DEFERRED] application/msword -> ../documents/doc/testRangeInsertion.doc
[DEFERRED] application/msword -> ../documents/doc/SimpleMacro.doc
[DEFERRED] application/msword -> ../documents/doc/Bug46610_1.doc
[DEFERRED] application/mswor

Traceback (most recent call last):
  File "/var/folders/v8/f7vcy2kn4tq3lw_bpdwqbs9m0000gn/T/ipykernel_47817/4185757274.py", line 162, in connect_loader
    loaded = loader.load()
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/langchain_community/document_loaders/word_document.py", line 61, in load
    page_content=docx2txt.process(self.file_path),
                 ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/docx2txt/docx2txt.py", line 88, in process
    text += xml2text(zipf.read(doc_xml))
            ~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/docx2txt/docx2txt.py", line 58, in xml2text
    root = ET.fromstring(xml)
  File "/Users/wonsik/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/xml/etree/ElementTree.py", line 1348, in XML
    parser.feed(text)
    ~~~~

['정상적으로 로드된 문서(Document) 개수: 134',
 '로더 실행 중 에러가 발생한 파일 개수: 13',
 'UnstructuredLoader로 분리(추후 처리)된 파일 개수: 159',
 '지원하지 않는 MIME 타입(라우터 미등록) 파일 개수: 22']

In [7]:
#spreadssheets/csv

docs, failed_files, deferred_files, unsupported_files = connect_loader('../spreadsheets/csv')

[
    f"정상적으로 로드된 문서(Document) 개수: {len(docs)}",
    f"로더 실행 중 에러가 발생한 파일 개수: {len(failed_files)}",
    f"UnstructuredLoader로 분리(추후 처리)된 파일 개수: {len(deferred_files)}",
    f"지원하지 않는 MIME 타입(라우터 미등록) 파일 개수: {len(unsupported_files)}",
]

[ERROR] 파일 로드 실패: ../spreadsheets/csv/candidate_visits.csv (RuntimeError) - Error loading ../spreadsheets/csv/candidate_visits.csv


Traceback (most recent call last):
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/langchain_community/document_loaders/csv_loader.py", line 135, in lazy_load
    yield from self.__read_file(csvfile)
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/langchain_community/document_loaders/csv_loader.py", line 155, in __read_file
    for i, row in enumerate(csv_reader):
                  ~~~~~~~~~^^^^^^^^^^^^
  File "/Users/wonsik/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/csv.py", line 178, in __next__
    row = next(self.reader)
  File "/Users/wonsik/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/encodings/ascii.py", line 26, in decode
    return codecs.ascii_decode(input, self.errors)[0]
           ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'ascii' codec can't decode byte 0xe2 in position 5776: ordinal not in range(128)

The above

[ERROR] 파일 로드 실패: ../spreadsheets/csv/data_aging_congress.csv (RuntimeError) - Error loading ../spreadsheets/csv/data_aging_congress.csv


Traceback (most recent call last):
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/langchain_community/document_loaders/csv_loader.py", line 135, in lazy_load
    yield from self.__read_file(csvfile)
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/langchain_community/document_loaders/csv_loader.py", line 155, in __read_file
    for i, row in enumerate(csv_reader):
                  ~~~~~~~~~^^^^^^^^^^^^
  File "/Users/wonsik/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/csv.py", line 178, in __next__
    row = next(self.reader)
  File "/Users/wonsik/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/encodings/ascii.py", line 26, in decode
    return codecs.ascii_decode(input, self.errors)[0]
           ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'ascii' codec can't decode byte 0xc3 in position 3647: ordinal not in range(128)

The above

[ERROR] 파일 로드 실패: ../spreadsheets/csv/online_weekly.csv (RuntimeError) - Error loading ../spreadsheets/csv/online_weekly.csv
[ERROR] 파일 로드 실패: ../spreadsheets/csv/daily_show_guests.csv (RuntimeError) - Error loading ../spreadsheets/csv/daily_show_guests.csv
[ERROR] 파일 로드 실패: ../spreadsheets/csv/classic-rock-raw-data.csv (RuntimeError) - Error loading ../spreadsheets/csv/classic-rock-raw-data.csv
[ERROR] 파일 로드 실패: ../spreadsheets/csv/congress-terms.csv (RuntimeError) - Error loading ../spreadsheets/csv/congress-terms.csv


...

[ERROR] 파일 로드 실패: ../spreadsheets/csv/impeachment_polls.csv (RuntimeError) - Error loading ../spreadsheets/csv/impeachment_polls.csv
[ERROR] 파일 로드 실패: ../spreadsheets/csv/fivethirtyeight_election_deniers.csv (RuntimeError) - Error loading ../spreadsheets/csv/fivethirtyeight_election_deniers.csv
[ERROR] 파일 로드 실패: ../spreadsheets/csv/fandango_score_comparison.csv (RuntimeError) - Error loading ../spreadsheets/csv/fandango_score_comparison.csv
[ERROR] 파일 로드 실패: ../spreadsheets/csv/actions_under_antiquities_act.csv (RuntimeError) - Error loading ../spreadsheets/csv/actions_under_antiquities_act.csv
[ERROR] 파일 로드 실패: ../spreadsheets/csv/cable_weekly.csv (RuntimeError) - Error loading ../spreadsheets/csv/cable_weekly.csv


Traceback (most recent call last):
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/langchain_community/document_loaders/csv_loader.py", line 135, in lazy_load
    yield from self.__read_file(csvfile)
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/langchain_community/document_loaders/csv_loader.py", line 155, in __read_file
    for i, row in enumerate(csv_reader):
                  ~~~~~~~~~^^^^^^^^^^^^
  File "/Users/wonsik/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/csv.py", line 178, in __next__
    row = next(self.reader)
  File "/Users/wonsik/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/encodings/ascii.py", line 26, in decode
    return codecs.ascii_decode(input, self.errors)[0]
           ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'ascii' codec can't decode byte 0xe2 in position 5315: ordinal not in range(128)

The above

[ERROR] 파일 로드 실패: ../spreadsheets/csv/candidate_visits_2024-01-11.csv (RuntimeError) - Error loading ../spreadsheets/csv/candidate_visits_2024-01-11.csv
[ERROR] 파일 로드 실패: ../spreadsheets/csv/tweets.csv (RuntimeError) - Error loading ../spreadsheets/csv/tweets.csv


Traceback (most recent call last):
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/langchain_community/document_loaders/csv_loader.py", line 135, in lazy_load
    yield from self.__read_file(csvfile)
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/langchain_community/document_loaders/csv_loader.py", line 155, in __read_file
    for i, row in enumerate(csv_reader):
                  ~~~~~~~~~^^^^^^^^^^^^
  File "/Users/wonsik/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/csv.py", line 178, in __next__
    row = next(self.reader)
  File "/Users/wonsik/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/encodings/ascii.py", line 26, in decode
    return codecs.ascii_decode(input, self.errors)[0]
           ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'ascii' codec can't decode byte 0xe2 in position 1116: ordinal not in range(128)

The above

['정상적으로 로드된 문서(Document) 개수: 342292',
 '로더 실행 중 에러가 발생한 파일 개수: 13',
 'UnstructuredLoader로 분리(추후 처리)된 파일 개수: 0',
 '지원하지 않는 MIME 타입(라우터 미등록) 파일 개수: 0']

In [3]:
#spreadssheets/xls

docs, failed_files, deferred_files, unsupported_files = connect_loader('../spreadsheets/xls')

[
    f"정상적으로 로드된 문서(Document) 개수: {len(docs)}",
    f"로더 실행 중 에러가 발생한 파일 개수: {len(failed_files)}",
    f"UnstructuredLoader로 분리(추후 처리)된 파일 개수: {len(deferred_files)}",
    f"지원하지 않는 MIME 타입(라우터 미등록) 파일 개수: {len(unsupported_files)}",
]

Traceback (most recent call last):
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/langchain_community/document_loaders/csv_loader.py", line 135, in lazy_load
    yield from self.__read_file(csvfile)
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/langchain_community/document_loaders/csv_loader.py", line 155, in __read_file
    for i, row in enumerate(csv_reader):
                  ~~~~~~~~~^^^^^^^^^^^^
  File "/Users/wonsik/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/csv.py", line 177, in __next__
    self.fieldnames
  File "/Users/wonsik/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/csv.py", line 164, in fieldnames
    self._fieldnames = next(self.reader)
                       ~~~~^^^^^^^^^^^^^
  File "<frozen codecs>", line 325, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xd0 in position 0: invalid continuation byte

The 

[ERROR] 파일 로드 실패: ../spreadsheets/xls/excelant.xls (RuntimeError) - Error loading ../spreadsheets/xls/excelant.xls
[ERROR] 파일 로드 실패: ../spreadsheets/xls/NewStyleConditionalFormattings.xls (RuntimeError) - Error loading ../spreadsheets/xls/NewStyleConditionalFormattings.xls
[ERROR] 파일 로드 실패: ../spreadsheets/xls/58760.xlsx (UnprocessableEntityError) - Not a valid XLSX file.


Traceback (most recent call last):
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/unstructured/partition/xlsx.py", line 193, in sheets
    office_file = OfficeFile(io.BytesIO(self._file_bytes))
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/msoffcrypto/__init__.py", line 49, in OfficeFile
    return OOXMLFile(file)
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/msoffcrypto/format/ooxml.py", line 179, in __init__
    raise exceptions.FileFormatError("Unsupported file format")
msoffcrypto.exceptions.FileFormatError: Unsupported file format

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/var/folders/v8/f7vcy2kn4tq3lw_bpdwqbs9m0000gn/T/ipykernel_65609/4185757274.py", line 162, in connect_loader
    loaded = loader.load()
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/

[ERROR] 파일 로드 실패: ../spreadsheets/xls/RepeatingRowsCols.xls (RuntimeError) - Error loading ../spreadsheets/xls/RepeatingRowsCols.xls
[ERROR] 파일 로드 실패: ../spreadsheets/xls/12843-1.xls (RuntimeError) - Error loading ../spreadsheets/xls/12843-1.xls
[ERROR] 파일 로드 실패: ../spreadsheets/xls/44010-TwoCharts.xls (RuntimeError) - Error loading ../spreadsheets/xls/44010-TwoCharts.xls
[ERROR] 파일 로드 실패: ../spreadsheets/xls/60273.xls (RuntimeError) - Error loading ../spreadsheets/xls/60273.xls
[ERROR] 파일 로드 실패: ../spreadsheets/xls/53691.xls (RuntimeError) - Error loading ../spreadsheets/xls/53691.xls
[ERROR] 파일 로드 실패: ../spreadsheets/xls/44010-SingleChart.xls (RuntimeError) - Error loading ../spreadsheets/xls/44010-SingleChart.xls
[ERROR] 파일 로드 실패: ../spreadsheets/xls/clusterfuzz-testcase-minimized-POIXSSFFuzzer-6123461607817216.xlsx (UnprocessableEntityError) - Not a valid XLSX file.
[UNSUPPORTED] application/x-ole-storage -> ../spreadsheets/xls/OddStyleRecord.xls
[ERROR] 파일 로드 실패: ../spreadsheets/x

Traceback (most recent call last):
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/langchain_community/document_loaders/csv_loader.py", line 135, in lazy_load
    yield from self.__read_file(csvfile)
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/langchain_community/document_loaders/csv_loader.py", line 155, in __read_file
    for i, row in enumerate(csv_reader):
                  ~~~~~~~~~^^^^^^^^^^^^
  File "/Users/wonsik/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/csv.py", line 177, in __next__
    self.fieldnames
  File "/Users/wonsik/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/csv.py", line 164, in fieldnames
    self._fieldnames = next(self.reader)
                       ~~~~^^^^^^^^^^^^^
  File "<frozen codecs>", line 325, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xd0 in position 0: invalid continuation byte

The 

...

No features in text.
Traceback (most recent call last):
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/langchain_community/document_loaders/csv_loader.py", line 135, in lazy_load
    yield from self.__read_file(csvfile)
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/langchain_community/document_loaders/csv_loader.py", line 155, in __read_file
    for i, row in enumerate(csv_reader):
                  ~~~~~~~~~^^^^^^^^^^^^
  File "/Users/wonsik/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/csv.py", line 177, in __next__
    self.fieldnames
  File "/Users/wonsik/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/csv.py", line 164, in fieldnames
    self._fieldnames = next(self.reader)
                       ~~~~^^^^^^^^^^^^^
  File "<frozen codecs>", line 325, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xd0 in position 0: invalid co

[ERROR] 파일 로드 실패: ../spreadsheets/xls/XRefCalc.xls (RuntimeError) - Error loading ../spreadsheets/xls/XRefCalc.xls
[ERROR] 파일 로드 실패: ../spreadsheets/xls/ex42564-21435.xls (RuntimeError) - Error loading ../spreadsheets/xls/ex42564-21435.xls
[ERROR] 파일 로드 실패: ../spreadsheets/xls/ex45978-extraLinkTableSheets.xls (RuntimeError) - Error loading ../spreadsheets/xls/ex45978-extraLinkTableSheets.xls
[ERROR] 파일 로드 실패: ../spreadsheets/xls/shared_formulas.xls (RuntimeError) - Error loading ../spreadsheets/xls/shared_formulas.xls
[ERROR] 파일 로드 실패: ../spreadsheets/xls/ColumnStyle1dp.xls (RuntimeError) - Error loading ../spreadsheets/xls/ColumnStyle1dp.xls
[ERROR] 파일 로드 실패: ../spreadsheets/xls/SimpleChart.xls (RuntimeError) - Error loading ../spreadsheets/xls/SimpleChart.xls
[ERROR] 파일 로드 실패: ../spreadsheets/xls/46137.xls (RuntimeError) - Error loading ../spreadsheets/xls/46137.xls
[ERROR] 파일 로드 실패: ../spreadsheets/xls/24215.xls (RuntimeError) - Error loading ../spreadsheets/xls/24215.xls
[ERROR] 파일

Traceback (most recent call last):
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/langchain_community/document_loaders/csv_loader.py", line 135, in lazy_load
    yield from self.__read_file(csvfile)
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/langchain_community/document_loaders/csv_loader.py", line 155, in __read_file
    for i, row in enumerate(csv_reader):
                  ~~~~~~~~~^^^^^^^^^^^^
  File "/Users/wonsik/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/csv.py", line 177, in __next__
    self.fieldnames
  File "/Users/wonsik/.local/share/uv/python/cpython-3.13.7-macos-aarch64-none/lib/python3.13/csv.py", line 164, in fieldnames
    self._fieldnames = next(self.reader)
                       ~~~~^^^^^^^^^^^^^
  File "<frozen codecs>", line 325, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xd0 in position 0: invalid continuation byte

The 

[DEFERRED] application/octet-stream -> ../spreadsheets/xls/SimpleEMF_mac.emf
[ERROR] 파일 로드 실패: ../spreadsheets/xls/30978-alt.xls (RuntimeError) - Error loading ../spreadsheets/xls/30978-alt.xls
[ERROR] 파일 로드 실패: ../spreadsheets/xls/56325a.xls (RuntimeError) - Error loading ../spreadsheets/xls/56325a.xls
[ERROR] 파일 로드 실패: ../spreadsheets/xls/45761.xls (RuntimeError) - Error loading ../spreadsheets/xls/45761.xls
[ERROR] 파일 로드 실패: ../spreadsheets/xls/42464-ExpPtg-ok.xls (RuntimeError) - Error loading ../spreadsheets/xls/42464-ExpPtg-ok.xls
[ERROR] 파일 로드 실패: ../spreadsheets/xls/countifExamples.xls (RuntimeError) - Error loading ../spreadsheets/xls/countifExamples.xls
[ERROR] 파일 로드 실패: ../spreadsheets/xls/PercentPtg.xls (RuntimeError) - Error loading ../spreadsheets/xls/PercentPtg.xls


: 

In [5]:
#presentations/ppt

docs, failed_files, deferred_files, unsupported_files = connect_loader('../presentations/ppt')

[
    f"정상적으로 로드된 문서(Document) 개수: {len(docs)}",
    f"로더 실행 중 에러가 발생한 파일 개수: {len(failed_files)}",
    f"UnstructuredLoader로 분리(추후 처리)된 파일 개수: {len(deferred_files)}",
    f"지원하지 않는 MIME 타입(라우터 미등록) 파일 개수: {len(unsupported_files)}",
]

[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/45776.ppt
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/alterman_security.ppt
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/bug58718_008558.ppt
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/42474-2.ppt
[DEFERRED] application/octet-stream -> ../presentations/ppt/bug65551.pptx
[ERROR] 파일 로드 실패: ../presentations/ppt/missing-blip-fill.pptx (ValueError) - no embedded image
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/WithComments.ppt


Traceback (most recent call last):
  File "/var/folders/v8/f7vcy2kn4tq3lw_bpdwqbs9m0000gn/T/ipykernel_47817/2276677592.py", line 163, in connect_loader
    loaded = loader.load()
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/langchain_core/document_loaders/base.py", line 32, in load
    return list(self.lazy_load())
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/langchain_docling/loader.py", line 117, in lazy_load
    conv_res = self._converter.convert(
        source=file_path,
        **self._convert_kwargs,
    )
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/pydantic/_internal/_validate_call.py", line 39, in wrapper_function
    return wrapper(*args, **kwargs)
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/pydantic/_internal/_validate_call.py", line 136, in __call__
    res = self.__pydantic_validato

[UNSUPPORTED] image/svg+xml -> ../presentations/ppt/pptx2svg.svg
[DEFERRED] application/octet-stream -> ../presentations/ppt/OverlappingRelations.pptx
[UNSUPPORTED] application/x-ole-storage -> ../presentations/ppt/clusterfuzz-testcase-minimized-POIHSLFFuzzer-6032591399288832.ppt
[UNSUPPORTED] application/x-ole-storage -> ../presentations/ppt/clusterfuzz-testcase-minimized-POIHSLFFuzzer-6614960949821440.ppt
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/42474-1.ppt
[UNSUPPORTED] application/x-ole-storage -> ../presentations/ppt/clusterfuzz-testcase-minimized-POIHSLFFuzzer-6710128412590080.ppt
[UNSUPPORTED] application/encrypted -> ../presentations/ppt/ppt_with_png_encrypted.ppt
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/missing_core_records.ppt
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/ParagraphStylesShorterThanCharStyles.ppt
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/br.com.tvcamboriu.www_pps_Pensar_5

An unexpected error occurred while opening the document Divino_Revelado.pptx
Traceback (most recent call last):
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/docling/backend/mspowerpoint_backend.py", line 50, in __init__
    self.pptx_obj = Presentation(self.path_or_stream)
                    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/pptx/api.py", line 31, in Presentation
    presentation_part = Package.open(pptx).main_document_part
                        ~~~~~~~~~~~~^^^^^^
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/pptx/opc/package.py", line 82, in open
    return cls(pkg_file)._load()
           ~~~~~~~~~~~~~~~~~~~^^
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/pptx/opc/package.py", line 160, in _load
    pkg_xml_rels, parts = _PackageLoader.load(self._pkg

[ERROR] 파일 로드 실패: ../presentations/ppt/Divino_Revelado.pptx (ConversionError) - Input document Divino_Revelado.pptx is not valid.
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/41071.ppt
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/bug57820-initTableNullRefrenceException.ppt
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/text-margins.ppt
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/54722.ppt
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/bug69697.ppt
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/incorrect_slide_order.ppt
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/clusterfuzz-testcase-minimized-POIHSLFFuzzer-4630915954114560.ppt
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/60294.ppt
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/bug58516.ppt
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/PPT95.ppt
[

...

HTTP Error 429 thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json
Retrying in 1s [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json
Retrying in 2s [Retry 2/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json
Retrying in 4s [Retry 3/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json
Retrying in 8s [Retry 4/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformer

[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/clusterfuzz-testcase-minimized-POIHSLFFuzzer-5306877435838464.ppt
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/56260.ppt
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/51731.ppt
[UNSUPPORTED] application/zip -> ../presentations/ppt/clusterfuzz-testcase-minimized-POIXSLFFuzzer-4986044400861184.pptx
[UNSUPPORTED] application/encrypted -> ../presentations/ppt/clusterfuzz-testcase-minimized-POIHSLFFuzzer-5231088823566336.ppt
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/54111.ppt
[UNSUPPORTED] image/tiff -> ../presentations/ppt/testtiff.tif
[UNSUPPORTED] image/wmf -> ../presentations/ppt/64716_image3.wmf
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/master_text.ppt
[UNSUPPORTED] image/wmf -> ../presentations/ppt/64716_image2.wmf
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/empty.ppt


An unexpected error occurred while opening the document clusterfuzz-testcase-minimized-POIXSLFFuzzer-5471515212382208.pptx
Traceback (most recent call last):
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/docling/backend/mspowerpoint_backend.py", line 50, in __init__
    self.pptx_obj = Presentation(self.path_or_stream)
                    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/pptx/api.py", line 31, in Presentation
    presentation_part = Package.open(pptx).main_document_part
                        ~~~~~~~~~~~~^^^^^^
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/pptx/opc/package.py", line 82, in open
    return cls(pkg_file)._load()
           ~~~~~~~~~~~~~~~~~~~^^
  File "/Users/wonsik/Desktop/Voyager/voyager-app-backend/.venv/lib/python3.13/site-packages/pptx/opc/package.py", line 160, in _load
    pkg_x

[ERROR] 파일 로드 실패: ../presentations/ppt/clusterfuzz-testcase-minimized-POIXSLFFuzzer-5471515212382208.pptx (ConversionError) - Input document clusterfuzz-testcase-minimized-POIXSLFFuzzer-5471515212382208.pptx is not valid.
[UNSUPPORTED] image/bmp -> ../presentations/ppt/clock.dib
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/bug56240.ppt
[DEFERRED] application/vnd.ms-powerpoint -> ../presentations/ppt/basic_test_ppt_file.ppt


['정상적으로 로드된 문서(Document) 개수: 470',
 '로더 실행 중 에러가 발생한 파일 개수: 13',
 'UnstructuredLoader로 분리(추후 처리)된 파일 개수: 124',
 '지원하지 않는 MIME 타입(라우터 미등록) 파일 개수: 36']

In [2]:
#rtf

docs, failed_files, deferred_files, unsupported_files = connect_loader('../texts/rtf')

[
    f"정상적으로 로드된 문서(Document) 개수: {len(docs)}",
    f"로더 실행 중 에러가 발생한 파일 개수: {len(failed_files)}",
    f"UnstructuredLoader로 분리(추후 처리)된 파일 개수: {len(deferred_files)}",
    f"지원하지 않는 MIME 타입(라우터 미등록) 파일 개수: {len(unsupported_files)}",
]

[WARNING] Unsupported code page 950. Text will likely be garbled.


[WARNING] Unsupported code page 0. Text will likely be garbled.


[WARNING] Unsupported code page 1250. Text will likely be garbled.


[WARNING] Unsupported code page 1250. Text will likely be garbled.


[WARNING] Unsupported code page 1250. Text will likely be garbled.


[WARNING] Unsupported code page 1251. Text will likely be garbled.


[WARNING] Unsupported code page 1250. Text will likely be garbled.


[WARNING] Unsupported code page 1250. Text will likely be garbled.


[WARNING] Unsupported code page 0. Text will likely be garbled.


[WARNING] Unsupported code page 950. Text will likely be garbled.


[WARNING] Unsupported code page 950. Text will likely be garbled.


[WARNING] Unsupported code page 950. Text will likely be garbled.


[WARNING] Unsupported code page 1251. Text will likely be garbled.


[WARNING] Unsupported code page 1251. Text will likely be garbled.




[UNSUPPORTED] image/png -> ../texts/rtf/libreoffice.png


[WARNING] Unsupported code page 1250. Text will likely be garbled.


[WARNING] Unsupported code page 1251. Text will likely be garbled.




[UNSUPPORTED] application/vnd.oasis.opendocument.text -> ../texts/rtf/fdo68291.odt


[WARNING] Unsupported code page 1250. Text will likely be garbled.




['정상적으로 로드된 문서(Document) 개수: 218',
 '로더 실행 중 에러가 발생한 파일 개수: 0',
 'UnstructuredLoader로 분리(추후 처리)된 파일 개수: 0',
 '지원하지 않는 MIME 타입(라우터 미등록) 파일 개수: 2']

In [3]:
#txt

docs, failed_files, deferred_files, unsupported_files = connect_loader('../texts/txt')

[
    f"정상적으로 로드된 문서(Document) 개수: {len(docs)}",
    f"로더 실행 중 에러가 발생한 파일 개수: {len(failed_files)}",
    f"UnstructuredLoader로 분리(추후 처리)된 파일 개수: {len(deferred_files)}",
    f"지원하지 않는 MIME 타입(라우터 미등록) 파일 개수: {len(unsupported_files)}",
]

[UNSUPPORTED] text/x-affix -> ../texts/txt/145.txt
[UNSUPPORTED] text/x-affix -> ../texts/txt/345.txt


['정상적으로 로드된 문서(Document) 개수: 96',
 '로더 실행 중 에러가 발생한 파일 개수: 0',
 'UnstructuredLoader로 분리(추후 처리)된 파일 개수: 0',
 '지원하지 않는 MIME 타입(라우터 미등록) 파일 개수: 2']

In [ ]:
# 정상적으로 Loader 연결된 파일들 내용 출력
contents = [doc.page_content for doc in docs]
contents

['Song Clean: Caught Up in You\nARTIST CLEAN: .38 Special\nRelease Year: 1982\nCOMBINED: Caught Up in You by .38 Special\nFirst?: 1\nYear?: 1\nPlayCount: 82\nF*G: 82',
 'Song Clean: Fantasy Girl\nARTIST CLEAN: .38 Special\nRelease Year: \nCOMBINED: Fantasy Girl by .38 Special\nFirst?: 1\nYear?: 0\nPlayCount: 3\nF*G: 0',
 'Song Clean: Hold On Loosely\nARTIST CLEAN: .38 Special\nRelease Year: 1981\nCOMBINED: Hold On Loosely by .38 Special\nFirst?: 1\nYear?: 1\nPlayCount: 85\nF*G: 85',
 "Song Clean: Rockin' Into the Night\nARTIST CLEAN: .38 Special\nRelease Year: 1980\nCOMBINED: Rockin' Into the Night by .38 Special\nFirst?: 1\nYear?: 1\nPlayCount: 18\nF*G: 18",
 'Song Clean: Art For Arts Sake\nARTIST CLEAN: 10cc\nRelease Year: 1975\nCOMBINED: Art For Arts Sake by 10cc\nFirst?: 1\nYear?: 1\nPlayCount: 1\nF*G: 1',
 'Song Clean: Kryptonite\nARTIST CLEAN: 3 Doors Down\nRelease Year: 2000\nCOMBINED: Kryptonite by 3 Doors Down\nFirst?: 1\nYear?: 1\nPlayCount: 13\nF*G: 13',
 'Song Clean: Loser\

# 더미데이터 테스트 결과
- 확장자 폴더별 소요시간
doc - 2.8s
csv - 4.1s
xls - 커널 중단
ppt - 1m 41.3s
rtf - 15.8s
txt - 1.6s